In [1]:
%pip install -q pgserver psycopg[binary] sqlalchemy pandas

import os
import shutil
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text


pg = pgserver.get_server('/tmp/hw4_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [2]:
import kagglehub


dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print(dataset_dir)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
/kaggle/input/brazilian-ecommerce


In [3]:
all_files = sorted(os.listdir(dataset_dir))

print("\nAll files in dataset:")
for file_name in all_files:
    print(" -", file_name)


files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
    'olist_order_reviews_raw': 'olist_order_reviews_dataset.csv',
}


# Check, copy and load required CSV files
data_dir = "data"
os.makedirs(data_dir, exist_ok=True)

total_size_mb = 0

for table_name, file_name in files.items():

    source_path = os.path.join(dataset_dir, file_name)
    destination_path = os.path.join(data_dir, file_name)

    if not os.path.exists(source_path):
        raise FileNotFoundError(
            f"Required file not found: {source_path}"
        )

    df = pd.read_csv(source_path)

    size_mb = os.path.getsize(source_path) / (1024 ** 2)
    total_size_mb += size_mb

    print(f"\n{file_name}")
    print(f"Raw table: {table_name}")
    print(f"Shape: {df.shape}")
    print(f"Size: {size_mb:.2f} MB")
    print("Columns:")
    print(list(df.columns))

    # Copy CSV to data/
    shutil.copy2(source_path, destination_path)

    # Load into PostgreSQL raw table
    df.to_sql(
        table_name,
        engine,
        if_exists="replace",
        index=False,
        chunksize=10_000
    )

    print(f"Copied to: {destination_path}")
    print(f"Loaded into PostgreSQL: {table_name}")


print("\n--------------------------------")
print(f"Total data/ size: {total_size_mb:.2f} MB")
print("--------------------------------")

print("\nFiles in data/:")
for file_name in sorted(os.listdir(data_dir)):
    print(f" - {file_name}")




All files in dataset:
 - olist_customers_dataset.csv
 - olist_geolocation_dataset.csv
 - olist_order_items_dataset.csv
 - olist_order_payments_dataset.csv
 - olist_order_reviews_dataset.csv
 - olist_orders_dataset.csv
 - olist_products_dataset.csv
 - olist_sellers_dataset.csv
 - product_category_name_translation.csv

olist_customers_dataset.csv
Raw table: olist_customers_raw
Shape: (99441, 5)
Size: 8.62 MB
Columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
Copied to: data/olist_customers_dataset.csv
Loaded into PostgreSQL: olist_customers_raw

olist_orders_dataset.csv
Raw table: olist_orders_raw
Shape: (99441, 8)
Size: 16.84 MB
Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
Copied to: data/olist_orders_dataset.csv
Loaded into PostgreSQL: olist_orders_raw

olist_order_items_dat

In [4]:
# ============================================================
# 1. Setup і typed-схема Olist
# ============================================================

drop_sql = """

DROP TABLE IF EXISTS hw4_customer_segments    CASCADE;
DROP TABLE IF EXISTS hw4_seller_score         CASCADE;
DROP TABLE IF EXISTS hw4_seller_alerts        CASCADE;
DROP TABLE IF EXISTS olist_customer_dim       CASCADE;
DROP TABLE IF EXISTS olist_order_reviews      CASCADE;
DROP TABLE IF EXISTS olist_order_items        CASCADE;
DROP TABLE IF EXISTS olist_orders             CASCADE;
DROP TABLE IF EXISTS olist_customers          CASCADE;
DROP TABLE IF EXISTS olist_products           CASCADE;
DROP TABLE IF EXISTS olist_sellers            CASCADE;

"""

typed_ddl = """

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE INDEX idx_olist_customers_unique_id
    ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

CREATE INDEX idx_olist_order_items_seller_order
    ON olist_order_items (seller_id, order_id);

CREATE TABLE olist_order_reviews (
    review_row_id            BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    review_id                TEXT,
    order_id                 TEXT NOT NULL REFERENCES olist_orders(order_id),
    review_score             SMALLINT CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title     TEXT,
    review_comment_message   TEXT,
    review_creation_date     TIMESTAMP,
    review_answer_timestamp  TIMESTAMP
);

CREATE INDEX idx_olist_order_reviews_order_id
    ON olist_order_reviews (order_id);

"""

with engine.begin() as conn:
    conn.exec_driver_sql(drop_sql)
    conn.exec_driver_sql(typed_ddl)

print("\nTyped schema created.")


typed_insert_sql = """

INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews (
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
)
SELECT
    review_id,
    order_id,
    review_score::SMALLINT,
    review_comment_title,
    review_comment_message,
    review_creation_date::TIMESTAMP,
    review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;

"""


with engine.begin() as conn:
    conn.exec_driver_sql(typed_insert_sql)

print("Raw -> typed loading completed.")


smoke_sql = """

SELECT
    'customers' AS table_name,
    (SELECT COUNT(*) FROM olist_customers_raw) AS raw_rows,
    (SELECT COUNT(*) FROM olist_customers) AS typed_rows

UNION ALL

SELECT
    'orders',
    (SELECT COUNT(*) FROM olist_orders_raw),
    (SELECT COUNT(*) FROM olist_orders)

UNION ALL

SELECT
    'order_items',
    (SELECT COUNT(*) FROM olist_order_items_raw),
    (SELECT COUNT(*) FROM olist_order_items)

UNION ALL

SELECT
    'products',
    (SELECT COUNT(*) FROM olist_products_raw),
    (SELECT COUNT(*) FROM olist_products)

UNION ALL

SELECT
    'sellers',
    (SELECT COUNT(*) FROM olist_sellers_raw),
    (SELECT COUNT(*) FROM olist_sellers)

UNION ALL

SELECT
    'reviews',
    (SELECT COUNT(*) FROM olist_order_reviews_raw),
    (SELECT COUNT(*) FROM olist_order_reviews)

ORDER BY table_name;

"""


with engine.connect() as conn:
    smoke_df = pd.read_sql(
        text(smoke_sql),
        conn
    )

display(smoke_df)




Typed schema created.
Raw -> typed loading completed.


,table_name,raw_rows,typed_rows
0,customers,99441,99441
1,order_items,112650,112650
2,orders,99441,99441
3,products,32951,32951
4,reviews,99224,99224
5,sellers,3095,3095


In [5]:
# ============================================================
# 2. DML-запити з PostgreSQL-фічами
# ============================================================

sql = """

-- ============================================================
-- 2.1 INSERT ... RETURNING
-- створення alert для продавця з найбільшою кількістю запізнень
-- ============================================================

DROP TABLE IF EXISTS hw4_seller_alerts CASCADE;

CREATE TABLE hw4_seller_alerts (
    alert_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    seller_id  TEXT NOT NULL REFERENCES olist_sellers(seller_id),
    alert_type TEXT NOT NULL CHECK (
        alert_type IN ('late_delivery', 'low_rating', 'data_quality')
    ),
    severity   SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
    details    TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
late_sellers AS (
    SELECT
        so.seller_id,
        COUNT(DISTINCT o.order_id) AS late_orders
    FROM seller_orders AS so
    JOIN olist_orders AS o
        ON o.order_id = so.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
      AND o.order_delivered_customer_date::DATE >
          o.order_estimated_delivery_date::DATE
    GROUP BY so.seller_id
    ORDER BY late_orders DESC, so.seller_id
    LIMIT 1
)
INSERT INTO hw4_seller_alerts (
    seller_id,
    alert_type,
    severity,
    details
)
SELECT
    seller_id,
    'late_delivery',
    4,
    'Seller has ' || late_orders ||
    ' late delivered orders in the loaded dataset'
FROM late_sellers
RETURNING
    alert_id,
    seller_id,
    alert_type,
    severity,
    created_at;


-- ============================================================
-- 2.2 INSERT ... ON CONFLICT DO UPDATE
-- idempotent upsert агрегованих seller-score
-- ============================================================

DROP TABLE IF EXISTS hw4_seller_score CASCADE;

CREATE TABLE hw4_seller_score (
    seller_id   TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
    avg_rating  NUMERIC(3, 2),
    n_reviews   INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
    tier        TEXT CHECK (
        tier IN ('gold', 'silver', 'bronze')
    ),
    updated_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT
        so.seller_id,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
        COUNT(rpo.order_id) AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
)
INSERT INTO hw4_seller_score (
    seller_id,
    avg_rating,
    n_reviews,
    tier
)
SELECT
    seller_id,
    avg_rating,
    n_reviews,
    CASE
        WHEN avg_rating IS NULL THEN NULL
        WHEN avg_rating >= 4.5 THEN 'gold'
        WHEN avg_rating >= 3.5 THEN 'silver'
        ELSE 'bronze'
    END AS tier
FROM seller_rating
ON CONFLICT (seller_id) DO UPDATE
SET avg_rating = EXCLUDED.avg_rating,
    n_reviews  = EXCLUDED.n_reviews,
    tier       = EXCLUDED.tier,
    updated_at = NOW()
RETURNING
    seller_id,
    avg_rating,
    n_reviews,
    tier,
    updated_at;


-- ============================================================
-- 2.3 UPDATE ... FROM
-- оновлення прапорця запізнення в olist_orders
-- ============================================================

ALTER TABLE olist_orders
    ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;

WITH delivery_flags AS (
    SELECT
        order_id,
        (
            order_delivered_customer_date IS NOT NULL
            AND order_estimated_delivery_date IS NOT NULL
            AND order_delivered_customer_date::DATE >
                order_estimated_delivery_date::DATE
        ) AS is_late_calc
    FROM olist_orders
)
UPDATE olist_orders AS o
SET is_late = f.is_late_calc
FROM delivery_flags AS f
WHERE f.order_id = o.order_id
RETURNING
    o.order_id,
    o.customer_id,
    o.is_late;


-- ============================================================
-- 2.4 DELETE ... RETURNING
-- контрольоване видалення застарілих alert-ів
-- ============================================================

DELETE FROM hw4_seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING
    alert_id,
    seller_id,
    alert_type,
    created_at;

"""


with engine.begin() as conn:
    conn.exec_driver_sql(sql)

print("Tasks 2 completed.")


Tasks 2 completed.


In [6]:
# ============================================================
# 3. CREATE TABLE hw4_customer_segments
# ============================================================

sql = """

-- ============================================================
-- Customer dimension
-- ============================================================

DROP TABLE IF EXISTS hw4_customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;

CREATE TABLE olist_customer_dim (
    customer_unique_id TEXT PRIMARY KEY,
    latest_customer_id TEXT REFERENCES olist_customers(customer_id),
    customer_state     CHAR(2),
    customer_city      TEXT
);

INSERT INTO olist_customer_dim (
    customer_unique_id,
    latest_customer_id,
    customer_state,
    customer_city
)
SELECT DISTINCT ON (c.customer_unique_id)
    c.customer_unique_id,
    c.customer_id AS latest_customer_id,
    c.customer_state,
    c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
ORDER BY
    c.customer_unique_id,
    o.order_purchase_timestamp DESC NULLS LAST,
    c.customer_id;


-- ============================================================
-- hw4_customer_segments DDL
-- ============================================================

CREATE TABLE hw4_customer_segments (
    segment_id          BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    customer_unique_id  TEXT NOT NULL,
    segment_name        TEXT NOT NULL
        CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),

    rfm_score           SMALLINT NOT NULL
        CHECK (rfm_score BETWEEN 1 AND 5),

    recency_days        INTEGER
        CHECK (recency_days IS NULL OR recency_days >= 0),

    monetary_value      NUMERIC(12, 2) NOT NULL DEFAULT 0
        CHECK (monetary_value >= 0),

    n_orders            INTEGER NOT NULL DEFAULT 0
        CHECK (n_orders >= 0),

    assigned_at         TIMESTAMPTZ NOT NULL DEFAULT NOW(),

    UNIQUE (customer_unique_id)
);

ALTER TABLE hw4_customer_segments
    ADD CONSTRAINT fk_hw4_customer_segments_customer_unique
    FOREIGN KEY (customer_unique_id)
    REFERENCES olist_customer_dim(customer_unique_id)
    ON DELETE CASCADE;


-- ============================================================
-- Insert data
-- ============================================================

WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(
            SUM(oi.price + oi.freight_value),
            0
        )::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp
),
customer_stats AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(
            SUM(ot.order_total),
            0
        )::NUMERIC(12, 2) AS monetary_value,
        MAX(ot.order_date) AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers AS c
        ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot
        ON ot.customer_id = c.customer_id
    GROUP BY
        c.customer_unique_id
),
segmented AS (
    SELECT
        cs.customer_unique_id,
        cs.n_orders,
        cs.monetary_value,

        CASE
            WHEN cs.last_order_date IS NULL THEN NULL
            ELSE (
                rd.as_of_date - cs.last_order_date
            )::INTEGER
        END AS recency_days,

        CASE
            WHEN cs.monetary_value >= 1000 THEN 'VIP'

            WHEN cs.last_order_date IS NOT NULL
                 AND (
                     rd.as_of_date - cs.last_order_date
                 )::INTEGER <= 90
                 AND cs.n_orders = 1
                THEN 'new'

            WHEN cs.n_orders >= 2
                 OR cs.monetary_value >= 100
                THEN 'regular'

            ELSE 'inactive'
        END AS segment_name,

        CASE
            WHEN cs.monetary_value >= 1000 THEN 5
            WHEN cs.monetary_value >= 500  THEN 4
            WHEN cs.n_orders >= 2          THEN 3
            WHEN cs.n_orders = 1           THEN 2
            ELSE 1
        END::SMALLINT AS rfm_score

    FROM customer_stats AS cs
    CROSS JOIN reference_date AS rd
)

INSERT INTO hw4_customer_segments (
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
)
SELECT
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
FROM segmented

ON CONFLICT (customer_unique_id) DO UPDATE
SET
    segment_name   = EXCLUDED.segment_name,
    rfm_score      = EXCLUDED.rfm_score,
    recency_days   = EXCLUDED.recency_days,
    monetary_value = EXCLUDED.monetary_value,
    n_orders       = EXCLUDED.n_orders,
    assigned_at    = NOW()

RETURNING
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders;


-- ============================================================
-- Constraint check
-- ============================================================

SELECT
    conname,
    contype,
    pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'hw4_customer_segments'::regclass
ORDER BY conname;

"""


with engine.begin() as conn:
    conn.exec_driver_sql(sql)

print("Tasks 3 completed.")


Tasks 3 completed.


In [7]:
# ============================================================
# 4. JOIN-запити з business-interpretation
# ============================================================

join_queries = {

"4.1. INNER JOIN: продавці та їхня середня вартість замовлення": """

SELECT
    s.seller_id,
    s.seller_state,
    COUNT(DISTINCT o.order_id) AS n_orders,
    SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS total_revenue,
    (
        SUM(oi.price + oi.freight_value)
        / NULLIF(COUNT(DISTINCT o.order_id), 0)
    )::NUMERIC(12, 2) AS avg_order_value
FROM olist_sellers AS s
INNER JOIN olist_order_items AS oi
    ON oi.seller_id = s.seller_id
INNER JOIN olist_orders AS o
    ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY
    s.seller_id,
    s.seller_state
HAVING COUNT(DISTINCT o.order_id) >= 5
ORDER BY avg_order_value DESC
LIMIT 10;

""",


"4.2. LEFT JOIN + COALESCE: кількість замовлень для кожного продавця": """

SELECT
    s.seller_id,
    s.seller_state,
    s.seller_city,
    COALESCE(COUNT(DISTINCT oi.order_id), 0) AS n_orders,
    COALESCE(SUM(oi.price), 0)::NUMERIC(12, 2) AS product_revenue,
    COALESCE(SUM(oi.freight_value), 0)::NUMERIC(12, 2) AS freight_revenue
FROM olist_sellers AS s
LEFT JOIN olist_order_items AS oi
    ON oi.seller_id = s.seller_id
GROUP BY
    s.seller_id,
    s.seller_state,
    s.seller_city
ORDER BY n_orders DESC, product_revenue DESC
LIMIT 20;

""",


"4.3. FULL OUTER JOIN: порівняння міст customers і sellers": """

WITH customer_cities AS (
    SELECT
        customer_city AS city,
        COUNT(*) AS n_customers
    FROM olist_customers
    GROUP BY customer_city
),
seller_cities AS (
    SELECT
        seller_city AS city,
        COUNT(*) AS n_sellers
    FROM olist_sellers
    GROUP BY seller_city
)
SELECT
    COALESCE(c.city, s.city) AS city,
    COALESCE(c.n_customers, 0) AS n_customers,
    COALESCE(s.n_sellers, 0) AS n_sellers,
    CASE
        WHEN c.city IS NOT NULL AND s.city IS NOT NULL
            THEN 'both'
        WHEN c.city IS NOT NULL
            THEN 'customers_only'
        ELSE 'sellers_only'
    END AS city_presence
FROM customer_cities AS c
FULL OUTER JOIN seller_cities AS s
    ON c.city = s.city
ORDER BY n_customers DESC NULLS LAST
LIMIT 20;

""",


"4.4. SELF JOIN: sellers з однаковим штатом і однаковим містом": """

SELECT
    s1.seller_id AS seller_a,
    s2.seller_id AS seller_b,
    s1.seller_state,
    s1.seller_city
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
    ON s1.seller_state = s2.seller_state
   AND s1.seller_city = s2.seller_city
   AND s1.seller_id < s2.seller_id
ORDER BY
    s1.seller_state,
    s1.seller_city,
    seller_a,
    seller_b
LIMIT 20;

""",


"4.5. Anti-join: customers без жодного order item": """

SELECT
    c.customer_id,
    c.customer_unique_id,
    c.customer_state,
    c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
LEFT JOIN olist_order_items AS oi
    ON oi.order_id = o.order_id
WHERE oi.order_id IS NULL
ORDER BY c.customer_state, c.customer_city
LIMIT 20;

"""
}


for business_question, query in join_queries.items():

    print("\n" + "=" * 90)
    print(business_question)
    print("=" * 90)

    with engine.connect() as conn:
        result = pd.read_sql(
            text(query),
            conn
        )

    display(result)



4.1. INNER JOIN: продавці та їхня середня вартість замовлення


,seller_id,seller_state,n_orders,total_revenue,avg_order_value
0,e2a1ac9bf33e5549a2a4f834e70df2f8,SP,5,15200.56,3040.11
1,039e6ad9dae79614493083e241147386,SP,5,13273.42,2654.68
2,59417c56835dd8e2e72f91f809cd4092,SP,18,37657.77,2092.10
3,40db9e9aa57f7bb151bcda6b0f9bdbb7,PE,12,23823.99,1985.33
4,5d378b73ab7dd6f0418d743e5dcb0bd1,SP,6,11612.70,1935.45
5,397c4d0c005b6f41f90098ac724e28cb,SP,5,8622.37,1724.47
6,2bf6a2c1e71bbd29a4ad64e6d3c3629f,RJ,30,51629.49,1720.98
7,8fb791022c1fc8909664f48ab7dc636d,SP,5,8483.35,1696.67
8,05feb94f19d094d4b0f9281f0b1d4c99,SP,6,10050.04,1675.01
9,cf8ab1616079e2793aa29d524df01bb1,SP,6,10031.56,1671.93



4.2. LEFT JOIN + COALESCE: кількість замовлень для кожного продавця


,seller_id,seller_state,seller_city,n_orders,product_revenue,freight_revenue
0,6560211a19b47992c3666cc44a7e94c0,SP,sao paulo,1854,123304.83,27960.94
1,4a3ca9315b744ce9f8e9374361493884,SP,ibitinga,1806,200472.92,35067.04
2,cc419e0650a3c5ba77189a1882b7556a,SP,santo andre,1706,104288.42,25668.99
3,1f50f920176fa81dab994f9023523100,SP,sao jose do rio preto,1404,106939.21,35165.77
4,da8622b14eb17ae2831f4ac5b9dab84a,SP,piracicaba,1314,160236.57,24955.75
5,955fee9216a65b617aa5c0531780ce60,SP,sao paulo,1287,135171.70,25430.98
6,7a67c85e85bb2ce8582c35f2203ad736,SP,sao paulo,1160,141745.53,20902.85
7,ea8482cd71df3c1969d7b9473ff13abc,SP,sao paulo,1146,37177.52,17544.77
8,4869f7a5dfa277a7dca6462dcf3b52b2,SP,guariba,1132,229472.63,20168.07
9,3d871de0142ce09b7081e2b9d1733cb1,SP,campo limpo paulista,1080,94914.20,22433.51



4.3. FULL OUTER JOIN: порівняння міст customers і sellers


,city,n_customers,n_sellers,city_presence
0,sao paulo,15540,694,both
1,rio de janeiro,6882,96,both
2,belo horizonte,2773,68,both
3,brasilia,2131,28,both
4,curitiba,1521,127,both
5,campinas,1444,41,both
6,porto alegre,1379,28,both
7,salvador,1245,7,both
8,guarulhos,1189,50,both
9,sao bernardo do campo,938,32,both



4.4. SELF JOIN: sellers з однаковим штатом і однаковим містом


,seller_a,seller_b,seller_state,seller_city
0,1444c08e64d55fb3c25f0f09c07ffcf2,53243585a1d6dc2643021fd1853d8905,BA,lauro de freitas
1,398cb257329ef7af7f1943a8974a3cbc,75d34ebb1bd0bd7dde40dd507b8169c3,BA,salvador
2,398cb257329ef7af7f1943a8974a3cbc,a3dd39f583bc80bd8c5901c95878921e,BA,salvador
3,398cb257329ef7af7f1943a8974a3cbc,b00af24704019bd2e1b335e70ad11f7c,BA,salvador
4,398cb257329ef7af7f1943a8974a3cbc,eb4a59a06b3948e851a7d7a830bcbc42,BA,salvador
5,398cb257329ef7af7f1943a8974a3cbc,f3055483058c84d75ca2a3314fb4deef,BA,salvador
6,398cb257329ef7af7f1943a8974a3cbc,fc59392d66ef99377e50356ee4f3b4e1,BA,salvador
7,75d34ebb1bd0bd7dde40dd507b8169c3,a3dd39f583bc80bd8c5901c95878921e,BA,salvador
8,75d34ebb1bd0bd7dde40dd507b8169c3,b00af24704019bd2e1b335e70ad11f7c,BA,salvador
9,75d34ebb1bd0bd7dde40dd507b8169c3,eb4a59a06b3948e851a7d7a830bcbc42,BA,salvador



4.5. Anti-join: customers без жодного order item


,customer_id,customer_unique_id,customer_state,customer_city
0,ff0340330283c0d05ea7281e67fa2e76,dc80a79483121fee90b0d2f53d1054f5,AL,maceio
1,885a9dace03d5ee78e1639818cf8ca76,b2d2a58afa879edda95e349f7390903d,AL,uniao dos palmares
2,6a8e32e45208a7e5bb55c5f84ef10e53,924ff59a67457b502f4b9cad742087d2,AM,manaus
3,70f4e83e21c469871d23f680673f2195,d08705f36082bb64ae0acc92fe8bee3c,BA,caetanos
4,7043701548d462c834782d596ac13466,6a02391b65b57634144fc612aa9f108e,BA,casa nova
5,50ca8ddb57748311869736678c8dde1e,c4bb14eb428bc80c837bc11e89d1ec6a,BA,esplanada
6,ef21c78da6a79f7c34a9e2ca1c0ae880,4f4cfd4238efefcaf3e6905ac8bb9e76,BA,guanambi
7,86ebcb053082767a38cca1338ea45084,56b0ca904be7b0eeb32b90627b820f1a,BA,ipira
8,8dc1b1a511cd62c200c437fb43d2246e,c46e73e2dcd78c07869a4772469e9d3e,BA,itabuna
9,24b0e2bd287e47d54d193e7bbb51103f,00172711b30d52eea8b313a7f2cced02,BA,jequie


## Завдання 4 — JOIN-запити з business-interpretation

### 4.1 INNER JOIN

**Business-question:** Які продавці мають найбільший середній чек за доставленими замовленнями?

**Interpretation:** Запит об'єднує три таблиці: sellers, order_items та orders. Він показує продавців із мінімум 5 delivered-замовленнями та сортує їх за середньою вартістю замовлення. Це допомагає знайти sellers із високим середнім revenue на order.


### 4.2 LEFT JOIN + COALESCE

**Business-question:** Які продавці мають замовлення, а які не мають жодного замовлення?

**Interpretation:** `LEFT JOIN` зберігає всіх sellers, навіть якщо для seller немає відповідних order items. `COALESCE` перетворює відсутні значення на 0. Це дозволяє створити прості seller-level features: кількість замовлень, revenue та freight revenue.


### 4.3 FULL OUTER JOIN

**Business-question:** у яких містах є і customers, і sellers, а які міста представлені тільки з одного боку?

**Interpretation:** `FULL OUTER JOIN` дозволяє побачити всі міста з обох наборів. Це корисно для перевірки географічного покриття marketplace. Значення `customers_only` означає, що в місті є customers, але немає sellers, а `sellers_only` — навпаки.


### 4.4 SELF JOIN

**Business-question:** Які sellers працюють в одному місті та можуть конкурувати за той самий локальний ринок?

**Interpretation:** Це `SELF JOIN`, тому що таблиця `olist_sellers` приєднується сама до себе. Запит знаходить пари sellers з одного штату та міста. Умова `s1.seller_id` < `s2.seller_id` запобігає дублюванню пар.
Такий результат можна використати для локального конкурентного аналізу sellers.


### 4.5 Anti-join

**Business-question:** Які customers не мають жодного товару у своїх замовленнях?

**Interpretation:** Це anti-join через `LEFT JOIN ... IS NULL`. Він знаходить customers, для яких немає пов'язаних order items. Такі записи можуть бути цікавими для data-quality аналізу, оскільки customer існує в customer table, але пов'язаних товарних позицій немає.



In [8]:
# ============================================================
# 5. GROUP BY, HAVING і агрегати
# ============================================================

q51 = """

SELECT
    p.product_category_name,
    COUNT(*) AS n_items,
    AVG(oi.price)::NUMERIC(10, 2) AS avg_price,
    MIN(oi.price)::NUMERIC(10, 2) AS min_price,
    MAX(oi.price)::NUMERIC(10, 2) AS max_price
FROM olist_order_items AS oi
JOIN olist_products AS p
    ON p.product_id = oi.product_id
WHERE p.product_category_name IS NOT NULL
GROUP BY p.product_category_name
HAVING COUNT(*) >= 100
ORDER BY avg_price DESC
LIMIT 15;

"""

with engine.connect() as conn:
    result = pd.read_sql(text(q51), conn)

print("5.1 Середня ціна товару за категорією")
display(result)


q52 = """

SELECT
    oi.seller_id,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    COUNT(*) AS n_items,
    SUM(oi.price)::NUMERIC(12, 2) AS product_revenue,
    AVG(oi.price)::NUMERIC(10, 2) AS avg_item_price
FROM olist_order_items AS oi
JOIN olist_orders AS o
    ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY oi.seller_id
HAVING COUNT(DISTINCT oi.order_id) >= 20
ORDER BY product_revenue DESC
LIMIT 15;

"""

with engine.connect() as conn:
    result = pd.read_sql(text(q52), conn)

print("5.2. Активність продавців")
display(result)


q53 = """

SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS total_spend,
    COUNT(DISTINCT p.product_category_name) AS n_categories,
    STRING_AGG(
        DISTINCT COALESCE(p.product_category_name, '(unknown)'),
        ', ' ORDER BY COALESCE(p.product_category_name, '(unknown)')
    ) AS categories
FROM olist_customers AS c
JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
JOIN olist_order_items AS oi
    ON oi.order_id = o.order_id
LEFT JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT o.order_id) >= 2
ORDER BY total_spend DESC
LIMIT 20;

"""

with engine.connect() as conn:
    result = pd.read_sql(text(q53), conn)

print("5.3. Категорії товарів, які купують customers")
display(result)


5.1 Середня ціна товару за категорією


,product_category_name,n_items,avg_price,min_price,max_price
0,pcs,203,1098.34,34.50,6729.00
1,eletrodomesticos_2,238,476.12,13.90,2350.00
2,agro_industria_e_comercio,212,342.12,12.99,2990.00
3,instrumentos_musicais,680,281.62,4.90,4399.87
4,eletroportateis,679,280.78,6.50,4799.00
5,telefonia_fixa,264,225.69,6.00,1790.00
6,construcao_ferramentas_seguranca,194,208.99,8.90,3099.90
7,relogios_presentes,5991,201.14,8.99,3999.90
8,climatizacao,297,185.27,10.90,1599.00
9,moveis_quarto,109,183.75,6.90,650.00


5.2. Активність продавців


,seller_id,n_orders,n_items,product_revenue,avg_item_price
0,4869f7a5dfa277a7dca6462dcf3b52b2,1124,1148,226987.93,197.72
1,53243585a1d6dc2643021fd1853d8905,348,400,217940.44,544.85
2,4a3ca9315b744ce9f8e9374361493884,1772,1949,196882.12,101.02
3,fa1c13f2614d7b5c4749cbc52fecda94,578,579,190917.14,329.74
4,7c67e1448b00f6e969d365cea6b010ab,973,1355,186570.05,137.69
5,7e93a43ef30c4f03f38b393420bc753a,319,322,165981.49,515.47
6,da8622b14eb17ae2831f4ac5b9dab84a,1311,1548,159816.87,103.24
7,7a67c85e85bb2ce8582c35f2203ad736,1145,1155,139658.69,120.92
8,1025f0e2d44d7041d6cf58b6550e0bfa,910,1420,138208.56,97.33
9,955fee9216a65b617aa5c0531780ce60,1261,1472,131836.71,89.56


5.3. Категорії товарів, які купують customers


,customer_unique_id,n_orders,total_spend,n_categories,categories
0,da122df9eeddfedc1dc1f5349a1a690c,2,7388.00,1,eletroportateis
1,c8460e4251689ba205045f3ea17884a1,4,4080.00,1,telefonia
2,055ec572ac7f3c7bdd04a183830ebe59,2,3999.98,1,automotivo
3,59d66d72939bc9497e19d89c61a96d5f,2,3459.00,1,esporte_lazer
4,58c1b085b54c03a1f1ab5f13d64c2b1c,2,2999.98,1,pcs
5,d77aa95864ae5b42160937615628723a,2,2400.00,1,pcs
6,7b0eaf68a16e4808e5388c67345033c9,2,2238.42,1,bebes
7,86df00dc5fd68f4dd5d5945ca19f3ed6,3,2200.00,1,informatica_acessorios
8,6ddbc64bd04d40f7768ff088d94cbeb8,2,2108.00,1,eletrodomesticos
9,73601b1eec55943e90ce8d61253d5c09,2,2009.98,1,consoles_games


## Завдання 5 — GROUP BY, HAVING і агрегати

### 5.1

**Business-question:** які категорії мають достатню кількість товарних позицій і найвищу середню ціну?

**Interpretation:** Запит показує категорії, які мають щонайменше 100 проданих item-ів. Для кожної категорії розраховується середня, мінімальна та максимальна ціна. `HAVING` використовується для відбору категорій після агрегації.


### 5.2

**Business-question:** які продавці мають щонайменше 20 замовлень і який у них загальний revenue?

**Interpretation:** Запит аналізує sellers тільки за delivered-замовленнями. `HAVING COUNT(DISTINCT oi.order_id) >= 20` залишає продавців із достатньою кількістю замовлень.  


### 5.3

**Business-question:** які категорії найчастіше зустрічаються в покупках реальних customers?

**Interpretation:** Запит показує customers, які зробили щонайменше 2 замовлення. Для кожного customer розраховується кількість замовлень, загальна сума витрат і кількість різних категорій. `STRING_AGG` об'єднує всі категорії customer в один рядок, а `HAVING` відбирає customers після агрегації.


In [9]:
# ============================================================
# 6. Set operations, підзапит/window
# ============================================================

q61 = """

SELECT customer_state AS state
FROM olist_customers

INTERSECT

SELECT seller_state AS state
FROM olist_sellers

ORDER BY state;

"""

with engine.connect() as conn:
    result = pd.read_sql(text(q61), conn)

print("6.1. Set operation — INTERSECT")
display(result)


q62 = """

WITH ranked_orders AS (
    SELECT
        c.customer_unique_id,
        o.order_id,
        o.order_purchase_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp DESC
        ) AS rn
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
)
SELECT
    customer_unique_id,
    order_id,
    order_purchase_timestamp
FROM ranked_orders
WHERE rn = 1
ORDER BY customer_unique_id
LIMIT 20;


"""

with engine.connect() as conn:
    result = pd.read_sql(text(q62), conn)

print("6.2. Window function — ROW_NUMBER()")
display(result)



6.1. Set operation — INTERSECT


,state
0,AC
1,AM
2,BA
3,CE
4,DF
5,ES
6,GO
7,MA
8,MG
9,MS


6.2. Window function — ROW_NUMBER()


,customer_unique_id,order_id,order_purchase_timestamp
0,0000366f3b9a7992bf8c76cfdf3221e2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,b33ec3b699337181488304f362a6b734,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,41272756ecddd9a9ed0180413cc22fb6,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,d957021f1127559cd947b62533f484f7,2017-11-14 19:45:42
5,0004bd2a26a76fe21f786e4fbd80607f,3e470077b690ea3e3d501cffb5e0c499,2018-04-05 19:33:16
6,00050ab1314c0e55a6ca13cf7181fecf,d0028facea13f508e880202d7097a5a1,2018-04-20 12:57:23
7,00053a61a98854899e70ed204dd4bafe,44e608f2db00c74a1fe329de44416a4e,2018-02-28 11:15:41
8,0005e1862207bf6ccc02e4228effd9a0,ae76bef74b97bcb0b3e355e60d9a6f9c,2017-03-04 23:32:12
9,0005ef4cd20d2893f0d9fbd94d3c0d97,01b330808c5819a6a3cb79b72f0b8288,2018-03-12 15:22:12



## Завдання 6 - Set operations, підзапит/window

### 6.1 INTERSECT

**Business-question:** у яких штатах є і customers, і sellers?

**Interpretation:** Використано `INTERSECT`, тому що потрібно знайти тільки ті штати, які присутні в обох наборах — серед customers і серед sellers. Результат містить кожен штат один раз.


### 6.2 Window function — ROW_NUMBER()

**Business-question:** яке є найновіше замовлення кожного реального покупця?

**Interpretation:** `ROW_NUMBER()` нумерує замовлення кожного customer від найновішого до найстарішого. `PARTITION BY customer_unique_id` створює окрему нумерацію для кожного реального покупця. `WHERE rn = 1` залишає тільки найновіше замовлення кожного customer.


## Reflection

1. Найскладнішим JOIN для мене був `FULL OUTER JOIN`, тому що потрібно враховувати три випадки: запис є в обох таблицях, тільки в першій або тільки в другій. У моєму запиті це допомогло визначити, які штати є серед customers і sellers, а які присутні тільки в одній групі.

2. Найскладніше було сформулювати business-question для `seller_score`. Review в Olist прив'язаний до order, а не безпосередньо до seller. Тому спочатку потрібно було отримати рівень `seller_id × order_id`, а review окремо агрегувати на рівні order. Це допомагає уникнути повторного підрахунку review через кілька товарів в одному замовленні.

3. Для production найбільш корисною PostgreSQL-фічею вважаю `ON CONFLICT`. Вона дозволяє повторно запускати запит без створення дублікатів. `RETURNING` також корисний, тому що дозволяє одразу побачити рядки, які були додані або змінені.

4. У реальному ML-проєкті я б використовувала window functions, щоб знаходити останню покупку customer, нумерувати його замовлення та отримувати інформацію про попередні покупки.

5. У Olist я помітила відсутні reviews, `NULL` категорії товарів, порожні review texts і кілька `customer_id`, які належать одному `customer_unique_id`.

6. Для customer-level ML-фіч використовується `customer_unique_id`, тому що він краще представляє реального покупця. Один покупець може мати кілька `customer_id`.

7. Перед розрахунком `seller_score` потрібно перейти до грануляності `seller_id × order_id`, щоб один review не враховувався декілька разів через кілька товарів в одному замовленні.   
